# Homework #2 (GR5069)

Sarayu Rao

In [0]:
from pyspark.sql.functions import col, round, avg, upper, substring, when, length, floor, datediff, current_date, max, min
import pyspark.sql.functions as F


In [0]:
df_pitstops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)

In [0]:
display(df_pitstops)

In [0]:
df_pitstops = df_pitstops.withColumns({"raceId": col("raceId").cast("int"),
                                      "driverId": col("driverId").cast("int"),
                                      "stop": col("stop").cast("int"),
                                      "lap": col("lap").cast("int"),
                                      "milliseconds": col("milliseconds").cast("int")})

In [0]:
display(df_pitstops)

In [0]:
df_avg_pit = df_pitstops.groupBy("raceId","driverId").avg("milliseconds")
df_avg_pit = df_avg_pit.orderBy("raceId")
display(df_avg_pit)

In [0]:
df_max_pit = df_pitstops.groupBy("raceId").max('milliseconds')
df_min_pit = df_pitstops.groupBy("raceId").min('milliseconds')
df_max_min_pit = df_max_pit.join(df_min_pit, on="raceId")
df_max_min_pit = df_max_min_pit.withColumnsRenamed({"max(milliseconds)": "Slowest Pit", "min(milliseconds)": "Fastest Pit"})
display(df_max_min_pit)

In [0]:
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
display(df_results)

In [0]:
df_results = df_results.withColumns({"raceId": col("raceId").cast("int"),
                                      "driverId": col("driverId").cast("int"),
                                      "resultId": col("resultId").cast("int"),
                                      "positionOrder": col("positionOrder").cast("int"),
                                     })

In [0]:
df_race_results = df_results["raceId","driverId","positionOrder"]
df_sorted_pit = df_avg_pit.join(df_race_results, on:= ["raceId","driverId"])
df_sorted_pit = df_sorted_pit.orderBy("raceId","positionOrder")
display(df_sorted_pit)

In [0]:
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
display(df_drivers)

In [0]:
print(df_drivers.filter(col("code") == "\\N").count()   )
df_drivers.filter(col("code").isNull()).count()   

In [0]:
driver_code = when(~(col("driverRef").contains("_")), F.upper(substring("driverRef", 1, 3))).when((col("driverRef").contains("_")) & (length(F.substring_index("driverRef","_",-1))< 3), F.upper(substring(F.substring_index("driverRef", "_", 1), 1, 3))).otherwise(F.upper(substring(F.substring_index("driverRef", "_", -1), 1, 3)))


#df_drivers = df_drivers.withColumn("code_new", when(col("code")=="\\N", F.upper(substring("driverRef", 0, 3))).otherwise(col("code")))
df_drivers = df_drivers.withColumn("code", 
                                   when((col("code")=="\\N"), driver_code)
                                   .otherwise(col("code")))


In [0]:
display(df_drivers)

In [0]:
df_drivers = df_drivers.withColumn('age', floor(datediff(current_date(), col('dob')) / 365))
display(df_drivers)

In [0]:
df_results = df_drivers.select("driverId", "age", "forename", "surname").join(df_results, on=["driverId"])
display(df_results)

In [0]:
# get oldest/youngest age per race
df_ages = df_results.groupBy("raceId").agg(max("age").alias("oldest_age"), min("age").alias("youngest_age"))
print(df_ages)

# join back to results to get the driverId for each
df_oldest = (df_ages.join(df_results, (df_ages.raceId == df_results.raceId) & (df_ages.oldest_age == df_results.age)).select(df_ages.raceId, "oldest_age", col("driverId").alias("oldest_driverId"), col("forename").alias("oldest_driverName"), col("surname")))
            
df_youngest = (df_ages.join(df_results, (df_ages.raceId == df_results.raceId) & (df_ages.youngest_age == df_results.age)).select(df_ages.raceId, "youngest_age", col("driverId").alias("youngest_driverId"),col("forename").alias("youngest_driverName"), col("surname")))

df_old_young_race = df_oldest.join(df_youngest, on="raceId")
display(df_old_young_race)

In [0]:
df_driver_standings = spark.read.csv("/Volumes/gr5069/raw/f1_data/driver_standings.csv", header=True)
display(df_driver_standings)


